https://unit8co.github.io/darts/examples/07-NBEATS-examples.html

In [1]:
import torch
import numpy as np
import pandas as pd
import shutil

from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller
from darts.metrics import mape, r2_score, mae, rmse
from darts import concatenate

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import warnings

warnings.filterwarnings("ignore")
import logging

logging.disable(logging.CRITICAL)

In [2]:
def display_forecast(pred_series, ts_transformed, start_date=None):
    plt.figure(figsize=(8, 5))
    if start_date:
        ts_transformed = ts_transformed.drop_before(start_date)
    ts_transformed.univariate_component(0).plot(label="actual")
    pred_series.plot(label=("predicted"))
    plt.title(
        "R2: {}\n".format(r2_score(ts_transformed.univariate_component(0), pred_series))
        + "MAPE: {}\n".format(mape(ts_transformed.univariate_component(0), pred_series))
        + "MAE: {}\n".format(mae(ts_transformed.univariate_component(0), pred_series))
        + "RMSE: {}\n".format(rmse(ts_transformed.univariate_component(0), pred_series))
    )
    plt.legend()

def display_forecast_plotly(title_text, pred_series, ts_transformed, start_date=None):

    if start_date:
        ts_transformed = ts_transformed.drop_before(start_date)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=ts_transformed.univariate_component(0).time_index, y=ts_transformed.univariate_component(0).pd_series(), name='actual'))
    fig.add_trace(go.Scatter(x=pred_series.time_index, y=pred_series.pd_series(), name='predicted'))

    # add title with R2, MAPE, MAE and RMSE
    fig.update_layout(title=f"{title_text}<br>R2: {r2_score(ts_transformed.univariate_component(0), pred_series)} | "
        + f"MAPE: {mape(ts_transformed.univariate_component(0), pred_series)} | "
        + f"MAE: {mae(ts_transformed.univariate_component(0), pred_series)} | "
        + f"RMSE: {rmse(ts_transformed.univariate_component(0), pred_series)}")

    fig.show()

## Get the Data & Process (merge)

In [3]:
# Load data
# df = pd.read_csv('../data/01-output-BTCUSDT_1d-from-2020-12-31 00:00:00-until-2022-12-31 00:00:00-log-return.csv')
df = pd.read_csv('../data/01-output-ETHUSDT_1d-from-2018-12-31 00:00:00-until-2020-12-31 00:00:00-log-return.csv')
# df = pd.read_csv('../data/01-output-SOLUSDT_1d-from-2020-12-31 00:00:00-until-2022-12-31 00:00:00-log-return.csv')


data_name = 'ETHUSDT_1h' # BTCUSDT_1h, ETHUSDT_1h, SOLUSDT_1h
from_date = '2019-05-31' # 2021-11-30, 2019-05-31, 2021-03-31
# until_date = '2019-05-31'

target_feature = 'processed_log_return_wtmra_0'

df_features = pd.read_csv(f'../data/02c-output-{data_name}-log-return-difference-and-change.csv')

In [4]:
df_features

,date,processed_log_return_wtmra_0,processed_log_return_wtmra_0_ROC_1_days,processed_log_return_wtmra_0_derivative_1_days,processed_log_return_wtmra_0_ROC_3_days,processed_log_return_wtmra_0_derivative_3_days,processed_log_return_wtmra_0_ROC_7_days,processed_log_return_wtmra_0_derivative_7_days,processed_log_return_wtmra_0_ROC_14_days,processed_log_return_wtmra_0_derivative_14_days,processed_log_return_wtmra_0_ROC_21_days,processed_log_return_wtmra_0_derivative_21_days,processed_log_return_wtmra_0_ROC_30_days,processed_log_return_wtmra_0_derivative_30_days
0,2019-01-30 00:00:00,0.084933,-1.959063,0.173491,-2.677865,0.135552,-1.607424,0.224757,-0.110121,-0.010510,-1.750068,0.198166,-0.494948,-0.083234
1,2019-01-30 01:00:00,-0.117713,0.624151,-0.045236,-46.844966,-0.120280,-4.110866,-0.155552,0.058628,-0.006519,-0.421252,0.085679,0.202939,-0.019858
2,2019-01-30 02:00:00,0.031340,-1.540547,0.089318,-2.745419,0.049296,-1.561380,0.087167,-0.606574,-0.048319,-0.922806,-0.374648,-0.640702,-0.055886
3,2019-01-30 03:00:00,0.077002,-0.429313,-0.057927,0.043296,0.003196,-0.358731,-0.043076,-1.441332,0.251479,-1.431293,0.255540,-1.211989,0.440240
4,2019-01-30 04:00:00,-0.050083,-0.487998,0.047735,1.356837,-0.028833,-0.420717,0.036374,-1.257365,-0.244680,-0.463207,0.043217,-1.141372,-0.404343
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25580,2021-12-30 20:00:00,-0.021361,-0.912697,0.223317,-0.782431,0.076821,-1.200945,-0.127666,-0.898722,0.189557,-1.183159,-0.137988,-0.940089,0.335187
25581,2021-12-30 21:00:00,-0.175831,-1.713723,-0.422189,-1.738563,-0.413903,0.286344,-0.039141,-2.280097,-0.313189,0.133989,-0.020776,-5.548919,-0.214485
25582,2021-12-30 22:00:00,0.283100,-2.719859,0.447706,-2.178902,0.523238,3.638617,0.222069,2.512330,0.202498,0.365638,0.075797,-6.248053,0.337044
25583,2021-12-30 23:00:00,-0.257809,-1.833010,-0.567300,-2.449872,-0.435624,1.414949,-0.151053,14.921127,-0.241616,-3.904794,-0.346562,-2.133400,-0.485274


In [5]:
# remove processed_log_return_wtmra_0 from features
df_features.drop(columns=['processed_log_return_wtmra_0'], inplace=True)

In [6]:
# Get the features columns into a list
list_features = df_features.columns.tolist()

# Remove 'date', 'processed_log_return_wtmra_0'
list_features.remove('date')
# list_features.remove(target_feature)

print(list_features)

['processed_log_return_wtmra_0_ROC_1_days', 'processed_log_return_wtmra_0_derivative_1_days', 'processed_log_return_wtmra_0_ROC_3_days', 'processed_log_return_wtmra_0_derivative_3_days', 'processed_log_return_wtmra_0_ROC_7_days', 'processed_log_return_wtmra_0_derivative_7_days', 'processed_log_return_wtmra_0_ROC_14_days', 'processed_log_return_wtmra_0_derivative_14_days', 'processed_log_return_wtmra_0_ROC_21_days', 'processed_log_return_wtmra_0_derivative_21_days', 'processed_log_return_wtmra_0_ROC_30_days', 'processed_log_return_wtmra_0_derivative_30_days']


In [7]:
# From df_features, get only the 00:00:00 rows
df_features = df_features[df_features['date'].str.contains('00:00:00')]

# Remove the 00:00:00 from the date column
df_features['date'] = df_features['date'].str.replace(' 00:00:00', '')

# inner join df (date) and df_features (end_date)
df_merged = df.merge(df_features, on='date', how='inner')

In [8]:
# Filter from_date 
df_merged = df_merged[df_merged['date'] >= from_date]
df_merged

,date,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return,normalized_outliers_processed_log_return,...,processed_log_return_wtmra_0_ROC_3_days,processed_log_return_wtmra_0_derivative_3_days,processed_log_return_wtmra_0_ROC_7_days,processed_log_return_wtmra_0_derivative_7_days,processed_log_return_wtmra_0_ROC_14_days,processed_log_return_wtmra_0_derivative_14_days,processed_log_return_wtmra_0_ROC_21_days,processed_log_return_wtmra_0_derivative_21_days,processed_log_return_wtmra_0_ROC_30_days,processed_log_return_wtmra_0_derivative_30_days
121,2019-05-31,268.92,288.62,240.14,254.56,1.066279e+06,254.56,-0.054543,-0.054543,-0.622462,...,-1.224173,-0.335553,0.601456,-0.023078,-1.082394,-0.807224,0.156477,-0.008314,-0.799165,0.244512
122,2019-06-01,254.59,268.72,245.21,267.90,6.021539e+05,267.90,0.051077,0.051077,0.490950,...,1.741980,0.135252,-9.475374,0.238014,-0.758872,-0.670017,-1.562272,0.591529,0.462337,0.067310
123,2019-06-02,267.90,275.50,260.68,264.33,4.556770e+05,264.33,-0.013415,-0.013415,-0.188912,...,0.399798,-0.118548,0.820802,-0.187109,0.881069,-0.194412,-0.280261,0.161624,4.957626,-0.345397
124,2019-06-03,264.33,273.20,263.20,268.88,3.015362e+05,268.88,0.017067,0.017067,0.132423,...,-1.750869,0.107586,-0.727182,-0.122981,2.565306,0.033198,-0.806278,-0.192032,-1.289734,0.205384
125,2019-06-04,268.87,270.00,248.00,249.91,3.973607e+05,249.91,-0.073164,-0.073164,-0.818766,...,-2.528865,-0.538383,-2.187452,-0.599594,-2.258663,-0.584086,-2.495066,-0.543196,6.664394,-0.283020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,2020-12-27,626.78,652.91,615.26,637.44,9.585855e+05,637.44,0.016801,0.016801,0.129618,...,-0.156308,-0.071423,3.524705,0.300311,-5.553237,0.470180,12.581206,0.357127,-1.883016,0.822099
698,2020-12-28,637.44,717.13,625.00,685.11,1.859968e+06,685.11,0.072119,0.072119,0.712768,...,3.193436,-0.422298,7.110938,-0.486168,-2.945065,-0.839636,-2.041729,-1.086860,-2.827898,-0.857911
699,2020-12-29,685.10,748.09,681.04,730.41,1.627154e+06,730.41,0.064027,0.064027,0.627458,...,1.268873,0.171205,-2.186087,0.564233,-2.650425,0.491618,-15.722754,0.326924,-2.673946,0.489011
700,2020-12-30,730.40,740.78,689.20,732.00,1.106876e+06,732.00,0.002174,0.002174,-0.024568,...,-1.440838,-0.555461,-3.885911,-0.228838,-2.829022,-0.262867,-1.481991,-0.522546,1.377787,-0.098475


In [9]:
df_merged[target_feature] = df_merged[target_feature].astype('float32')

for feature in list_features:
    df_merged[feature] = df_merged[feature].astype('float32')

In [10]:
df_merged.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'original_close',
       'processed_log_return', 'outliers_processed_log_return',
       'normalized_outliers_processed_log_return',
       'processed_log_return_wtmra_0', 'processed_log_return_wtmra_1',
       'processed_log_return_wtmra_2', 'processed_log_return_wtmra_3',
       'processed_log_return_wtmra_4', 'processed_log_return_wtmra_5',
       'processed_log_return_wtmra_5_4', 'processed_log_return_wtmra_5_4_3',
       'processed_log_return_wtmra_5_4_3_2',
       'processed_log_return_wtmra_5_4_3_2_1',
       'processed_log_return_wtmra_5_4_3_2_1_0',
       'processed_log_return_wtmra_0_1', 'processed_log_return_wtmra_0_1_2',
       'processed_log_return_wtmra_0_1_2_3',
       'processed_log_return_wtmra_0_1_2_3_4',
       'processed_log_return_wtmra_0_1_2_3_4_5',
       'processed_log_return_wtmra_0_ROC_1_days',
       'processed_log_return_wtmra_0_derivative_1_days',
       'processed_log_return_wtmra_0_ROC_3_days',
    

In [11]:
# Filter df_merged by a <= certain date
df_merged = df_merged[df_merged['date'] <= '2020-06-30'] # 2022-12-31, 2020-06-30, 2022-04-30

## Create the `series` object

In [12]:
# Convert to TimeSeries object
series_log_return = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=[target_feature])

series_processed_log_return_wtmra_0_ROC_1_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_ROC_1_days'])
series_processed_log_return_wtmra_0_derivative_1_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_derivative_1_days'])

series_processed_log_return_wtmra_0_ROC_3_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_ROC_3_days'])
series_processed_log_return_wtmra_0_derivative_3_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_derivative_3_days'])

series_processed_log_return_wtmra_0_ROC_7_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_ROC_7_days'])
series_processed_log_return_wtmra_0_derivative_7_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_derivative_7_days'])

series_processed_log_return_wtmra_0_ROC_14_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_ROC_14_days'])
series_processed_log_return_wtmra_0_derivative_14_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_derivative_14_days'])

series_processed_log_return_wtmra_0_ROC_30_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_ROC_30_days'])
series_processed_log_return_wtmra_0_derivative_30_days = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_derivative_30_days'])

## Split the data into train, validation and test sets

In [13]:
train_date_cut = "20200430" # 20221031, 20200430, 20220228
val_date_cut = "20200531" # 20221130, 20200531, 20220331

train_log_return, temp_log_return = series_log_return.split_after(pd.Timestamp(train_date_cut))
val_log_return, test_log_return = temp_log_return.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_ROC_1_days, temp_processed_log_return_wtmra_0_ROC_1_days = series_processed_log_return_wtmra_0_ROC_1_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_ROC_1_days, test_processed_log_return_wtmra_0_ROC_1_days = temp_processed_log_return_wtmra_0_ROC_1_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_derivative_1_days, temp_processed_log_return_wtmra_0_derivative_1_days = series_processed_log_return_wtmra_0_derivative_1_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_derivative_1_days, test_processed_log_return_wtmra_0_derivative_1_days = temp_processed_log_return_wtmra_0_derivative_1_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_ROC_3_days, temp_processed_log_return_wtmra_0_ROC_3_days = series_processed_log_return_wtmra_0_ROC_3_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_ROC_3_days, test_processed_log_return_wtmra_0_ROC_3_days = temp_processed_log_return_wtmra_0_ROC_3_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_derivative_3_days, temp_processed_log_return_wtmra_0_derivative_3_days = series_processed_log_return_wtmra_0_derivative_3_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_derivative_3_days, test_processed_log_return_wtmra_0_derivative_3_days = temp_processed_log_return_wtmra_0_derivative_3_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_ROC_7_days, temp_processed_log_return_wtmra_0_ROC_7_days = series_processed_log_return_wtmra_0_ROC_7_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_ROC_7_days, test_processed_log_return_wtmra_0_ROC_7_days = temp_processed_log_return_wtmra_0_ROC_7_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_derivative_7_days, temp_processed_log_return_wtmra_0_derivative_7_days = series_processed_log_return_wtmra_0_derivative_7_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_derivative_7_days, test_processed_log_return_wtmra_0_derivative_7_days = temp_processed_log_return_wtmra_0_derivative_7_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_ROC_14_days, temp_processed_log_return_wtmra_0_ROC_14_days = series_processed_log_return_wtmra_0_ROC_14_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_ROC_14_days, test_processed_log_return_wtmra_0_ROC_14_days = temp_processed_log_return_wtmra_0_ROC_14_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_derivative_14_days, temp_processed_log_return_wtmra_0_derivative_14_days = series_processed_log_return_wtmra_0_derivative_14_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_derivative_14_days, test_processed_log_return_wtmra_0_derivative_14_days = temp_processed_log_return_wtmra_0_derivative_14_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_ROC_30_days, temp_processed_log_return_wtmra_0_ROC_30_days = series_processed_log_return_wtmra_0_ROC_30_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_ROC_30_days, test_processed_log_return_wtmra_0_ROC_30_days = temp_processed_log_return_wtmra_0_ROC_30_days.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_derivative_30_days, temp_processed_log_return_wtmra_0_derivative_30_days = series_processed_log_return_wtmra_0_derivative_30_days.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_derivative_30_days, test_processed_log_return_wtmra_0_derivative_30_days = temp_processed_log_return_wtmra_0_derivative_30_days.split_after(pd.Timestamp(val_date_cut))

In [14]:
# train_log_return.plot(label="train")
# val_log_return.plot(label="val")
# test_log_return.plot(label="test")

# Make the train, val and test sets plot using plotly go


fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=val_log_return.time_index, y=val_log_return.pd_series(), line=dict(color="red"), name='val'))
fig.add_trace(go.Scatter(x=test_log_return.time_index, y=test_log_return.pd_series(), line=dict(color="red"), name='val'))

fig.update_layout(title=f'Train, val and test sets for feature: {target_feature}')

fig.show()

## Scale the data

In [15]:
# Normalize
scaler_processed_log_return_wtmra_0_ROC_1_days = Scaler()
scaler_processed_log_return_wtmra_0_derivative_1_days = Scaler()

scaler_processed_log_return_wtmra_0_ROC_3_days = Scaler()
scaler_processed_log_return_wtmra_0_derivative_3_days = Scaler()

scaler_processed_log_return_wtmra_0_ROC_7_days = Scaler()
scaler_processed_log_return_wtmra_0_derivative_7_days = Scaler()

scaler_processed_log_return_wtmra_0_ROC_14_days = Scaler()
scaler_processed_log_return_wtmra_0_derivative_14_days = Scaler()

scaler_processed_log_return_wtmra_0_ROC_30_days = Scaler()
scaler_processed_log_return_wtmra_0_derivative_30_days = Scaler()

train_processed_log_return_wtmra_0_ROC_1_days_scaled = scaler_processed_log_return_wtmra_0_ROC_1_days.fit_transform(train_processed_log_return_wtmra_0_ROC_1_days)
val_processed_log_return_wtmra_0_ROC_1_days_scaled = scaler_processed_log_return_wtmra_0_ROC_1_days.transform(val_processed_log_return_wtmra_0_ROC_1_days)
test_processed_log_return_wtmra_0_ROC_1_days_scaled = scaler_processed_log_return_wtmra_0_ROC_1_days.transform(test_processed_log_return_wtmra_0_ROC_1_days)

train_processed_log_return_wtmra_0_derivative_1_days_scaled = scaler_processed_log_return_wtmra_0_derivative_1_days.fit_transform(train_processed_log_return_wtmra_0_derivative_1_days)
val_processed_log_return_wtmra_0_derivative_1_days_scaled = scaler_processed_log_return_wtmra_0_derivative_1_days.transform(val_processed_log_return_wtmra_0_derivative_1_days)
test_processed_log_return_wtmra_0_derivative_1_days_scaled = scaler_processed_log_return_wtmra_0_derivative_1_days.transform(test_processed_log_return_wtmra_0_derivative_1_days)

train_processed_log_return_wtmra_0_ROC_3_days_scaled = scaler_processed_log_return_wtmra_0_ROC_3_days.fit_transform(train_processed_log_return_wtmra_0_ROC_3_days)
val_processed_log_return_wtmra_0_ROC_3_days_scaled = scaler_processed_log_return_wtmra_0_ROC_3_days.transform(val_processed_log_return_wtmra_0_ROC_3_days)
test_processed_log_return_wtmra_0_ROC_3_days_scaled = scaler_processed_log_return_wtmra_0_ROC_3_days.transform(test_processed_log_return_wtmra_0_ROC_3_days)

train_processed_log_return_wtmra_0_derivative_3_days_scaled = scaler_processed_log_return_wtmra_0_derivative_3_days.fit_transform(train_processed_log_return_wtmra_0_derivative_3_days)
val_processed_log_return_wtmra_0_derivative_3_days_scaled = scaler_processed_log_return_wtmra_0_derivative_3_days.transform(val_processed_log_return_wtmra_0_derivative_3_days)
test_processed_log_return_wtmra_0_derivative_3_days_scaled = scaler_processed_log_return_wtmra_0_derivative_3_days.transform(test_processed_log_return_wtmra_0_derivative_3_days)

train_processed_log_return_wtmra_0_ROC_7_days_scaled = scaler_processed_log_return_wtmra_0_ROC_7_days.fit_transform(train_processed_log_return_wtmra_0_ROC_7_days)
val_processed_log_return_wtmra_0_ROC_7_days_scaled = scaler_processed_log_return_wtmra_0_ROC_7_days.transform(val_processed_log_return_wtmra_0_ROC_7_days)
test_processed_log_return_wtmra_0_ROC_7_days_scaled = scaler_processed_log_return_wtmra_0_ROC_7_days.transform(test_processed_log_return_wtmra_0_ROC_7_days)

train_processed_log_return_wtmra_0_derivative_7_days_scaled = scaler_processed_log_return_wtmra_0_derivative_7_days.fit_transform(train_processed_log_return_wtmra_0_derivative_7_days)
val_processed_log_return_wtmra_0_derivative_7_days_scaled = scaler_processed_log_return_wtmra_0_derivative_7_days.transform(val_processed_log_return_wtmra_0_derivative_7_days)
test_processed_log_return_wtmra_0_derivative_7_days_scaled = scaler_processed_log_return_wtmra_0_derivative_7_days.transform(test_processed_log_return_wtmra_0_derivative_7_days)

train_processed_log_return_wtmra_0_ROC_14_days_scaled = scaler_processed_log_return_wtmra_0_ROC_14_days.fit_transform(train_processed_log_return_wtmra_0_ROC_14_days)
val_processed_log_return_wtmra_0_ROC_14_days_scaled = scaler_processed_log_return_wtmra_0_ROC_14_days.transform(val_processed_log_return_wtmra_0_ROC_14_days)
test_processed_log_return_wtmra_0_ROC_14_days_scaled = scaler_processed_log_return_wtmra_0_ROC_14_days.transform(test_processed_log_return_wtmra_0_ROC_14_days)

train_processed_log_return_wtmra_0_derivative_14_days_scaled = scaler_processed_log_return_wtmra_0_derivative_14_days.fit_transform(train_processed_log_return_wtmra_0_derivative_14_days)
val_processed_log_return_wtmra_0_derivative_14_days_scaled = scaler_processed_log_return_wtmra_0_derivative_14_days.transform(val_processed_log_return_wtmra_0_derivative_14_days)
test_processed_log_return_wtmra_0_derivative_14_days_scaled = scaler_processed_log_return_wtmra_0_derivative_14_days.transform(test_processed_log_return_wtmra_0_derivative_14_days)

train_processed_log_return_wtmra_0_ROC_30_days_scaled = scaler_processed_log_return_wtmra_0_ROC_30_days.fit_transform(train_processed_log_return_wtmra_0_ROC_30_days)
val_processed_log_return_wtmra_0_ROC_30_days_scaled = scaler_processed_log_return_wtmra_0_ROC_30_days.transform(val_processed_log_return_wtmra_0_ROC_30_days)
test_processed_log_return_wtmra_0_ROC_30_days_scaled = scaler_processed_log_return_wtmra_0_ROC_30_days.transform(test_processed_log_return_wtmra_0_ROC_30_days)

train_processed_log_return_wtmra_0_derivative_30_days_scaled = scaler_processed_log_return_wtmra_0_derivative_30_days.fit_transform(train_processed_log_return_wtmra_0_derivative_30_days)
val_processed_log_return_wtmra_0_derivative_30_days_scaled = scaler_processed_log_return_wtmra_0_derivative_30_days.transform(val_processed_log_return_wtmra_0_derivative_30_days)
test_processed_log_return_wtmra_0_derivative_30_days_scaled = scaler_processed_log_return_wtmra_0_derivative_30_days.transform(test_processed_log_return_wtmra_0_derivative_30_days)

## Create `past_covariates`

In [16]:
train_past_covariates = concatenate([train_processed_log_return_wtmra_0_ROC_1_days_scaled, train_processed_log_return_wtmra_0_derivative_1_days_scaled, train_processed_log_return_wtmra_0_ROC_3_days_scaled, train_processed_log_return_wtmra_0_derivative_3_days_scaled, train_processed_log_return_wtmra_0_ROC_7_days_scaled, train_processed_log_return_wtmra_0_derivative_7_days_scaled, train_processed_log_return_wtmra_0_ROC_14_days_scaled, train_processed_log_return_wtmra_0_derivative_14_days_scaled, train_processed_log_return_wtmra_0_ROC_30_days_scaled, train_processed_log_return_wtmra_0_derivative_30_days_scaled], axis=1)
val_past_covariates = concatenate([val_processed_log_return_wtmra_0_ROC_1_days_scaled, val_processed_log_return_wtmra_0_derivative_1_days_scaled, val_processed_log_return_wtmra_0_ROC_3_days_scaled, val_processed_log_return_wtmra_0_derivative_3_days_scaled, val_processed_log_return_wtmra_0_ROC_7_days_scaled, val_processed_log_return_wtmra_0_derivative_7_days_scaled, val_processed_log_return_wtmra_0_ROC_14_days_scaled, val_processed_log_return_wtmra_0_derivative_14_days_scaled, val_processed_log_return_wtmra_0_ROC_30_days_scaled, val_processed_log_return_wtmra_0_derivative_30_days_scaled], axis=1)
test_past_covariates = concatenate([test_processed_log_return_wtmra_0_ROC_1_days_scaled, test_processed_log_return_wtmra_0_derivative_1_days_scaled, test_processed_log_return_wtmra_0_ROC_3_days_scaled, test_processed_log_return_wtmra_0_derivative_3_days_scaled, test_processed_log_return_wtmra_0_ROC_7_days_scaled, test_processed_log_return_wtmra_0_derivative_7_days_scaled, test_processed_log_return_wtmra_0_ROC_14_days_scaled, test_processed_log_return_wtmra_0_derivative_14_days_scaled, test_processed_log_return_wtmra_0_ROC_30_days_scaled, test_processed_log_return_wtmra_0_derivative_30_days_scaled], axis=1)
past_covariates = concatenate([train_past_covariates, val_past_covariates, test_past_covariates], axis=0)

## Training and validade Univariate model

In [17]:
window_size = 7 
num_forecast_points = 1
df_metrics = pd.DataFrame()

In [18]:
uni_model_nbeats = NBEATSModel(
    input_chunk_length=window_size,
    output_chunk_length=num_forecast_points,
    batch_size= 2 * (window_size + num_forecast_points),
    random_state=0,
    n_epochs=100,
    num_layers=2,
    layer_widths=512,
    loss_fn=torch.nn.MSELoss(),
)

In [19]:
uni_model_nbeats.fit(train_log_return,
                    val_series=val_log_return, 
                    verbose=False)

NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=2, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=7, output_chunk_length=1, batch_size=16, random_state=0, n_epochs=100, loss_fn=MSELoss())

In [20]:
# concatenate test and val series
test_val_log_return = concatenate([val_log_return, test_log_return], axis=0)

In [21]:
pred = uni_model_nbeats.predict(n=(len(val_log_return) + len(test_log_return)))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=test_val_log_return.time_index, y=test_val_log_return.pd_series(), name='test'))
fig.add_trace(go.Scatter(x=pred.time_index, y=pred.pd_series(), name='forecast'))

# add title with R2, MAPE, MAE and RMSE
fig.update_layout(title="Univariate Results<br>Evaluation Metrics: R2: {}\n".format(r2_score(series_log_return, pred))
    + " MAPE: {}".format(mape(series_log_return, pred))
    + " MAE: {}".format(mae(series_log_return, pred))
    + " RMSE: {}".format(rmse(series_log_return, pred)))

fig.show()

# series_log_return.plot(label="actual")
# pred.plot(label="forecast")
# plt.legend()
# print("MAPE = {:.2f}%".format(mape(series_log_return, pred)))
# print("MAE = {:.5f}".format(mae(series_log_return, pred)))
# print("RMSE = {:.5f}".format(rmse(series_log_return, pred)))
# print("R2 = {:.2f}".format(r2_score(series_log_return, pred)))

Predicting: |          | 0/? [00:00<?, ?it/s]

In [22]:
# uni_pred_series_log_return = uni_model_nbeats.historical_forecasts(
#     test_log_return,
#     forecast_horizon=num_forecast_points,
#     stride=1,
#     retrain=False,
#     verbose=False,
# )

In [23]:
# display_forecast_plotly("Univariate", uni_pred_series_log_return, test_log_return)

In [24]:
# Export a .json with the CCY, scenario, R2, MAPE, MAE and RMSE
dict_metrics = {
    "CCY": data_name,
    "scenario": "difference-and-change",
    "type": "univariate",
    "R2": r2_score(series_log_return, pred),
    "MAPE": mape(series_log_return, pred),
    "MAE": mae(series_log_return, pred),
    "RMSE": rmse(series_log_return, pred)
}

# Append a dict to the df_metrics
df_metrics = df_metrics._append(dict_metrics, ignore_index=True)

## Training and validade Multivariate model

In [25]:
multi_model_nbeats = NBEATSModel(
    input_chunk_length=window_size,
    output_chunk_length=num_forecast_points,
    batch_size= 2 * (window_size + num_forecast_points),
    random_state=0,
    n_epochs=100,
    num_layers=2,
    layer_widths=512,
    loss_fn=torch.nn.MSELoss(),
)

In [26]:
# fit using multiple (two) target series
multi_model_nbeats.fit(train_log_return,
          val_series=val_log_return,
          past_covariates=train_past_covariates,
          val_past_covariates=val_past_covariates,
          verbose=False
          )

NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=2, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=7, output_chunk_length=1, batch_size=16, random_state=0, n_epochs=100, loss_fn=MSELoss())

In [27]:
pred = multi_model_nbeats.predict(n=(len(val_log_return) + len(test_log_return)), series=train_log_return, past_covariates=past_covariates)

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=test_val_log_return.time_index, y=test_val_log_return.pd_series(), name='test'))
fig.add_trace(go.Scatter(x=pred.time_index, y=pred.pd_series(), name='forecast'))

# add title with R2, MAPE, MAE and RMSE
fig.update_layout(title="Multivariate Results<br>Evaluation Metrics: R2: {}\n".format(r2_score(series_log_return, pred))
    + " MAPE: {}".format(mape(series_log_return, pred))
    + " MAE: {}".format(mae(series_log_return, pred))
    + " RMSE: {}".format(rmse(series_log_return, pred)))

fig.show()

# series_log_return.plot(label="actual")
# pred.plot(label="forecast")
# plt.legend()
# print("MAPE = {:.2f}%".format(mape(series_log_return, pred)))
# print("MAE = {:.5f}".format(mae(series_log_return, pred)))
# print("RMSE = {:.5f}".format(rmse(series_log_return, pred)))
# print("R2 = {:.2f}".format(r2_score(series_log_return, pred)))

Predicting: |          | 0/? [00:00<?, ?it/s]

In [28]:
# multi_pred_series_log_return = multi_model_nbeats.historical_forecasts(
#     test_log_return,
#     past_covariates=test_past_covariates,
#     forecast_horizon=num_forecast_points,
#     stride=1,
#     retrain=False,
#     verbose=False,
# )

In [29]:
# display_forecast_plotly("Multivariate", multi_pred_series_log_return, test_log_return)

In [30]:
# Export a .json with the CCY, scenario, R2, MAPE, MAE and RMSE
dict_metrics = {
    "CCY": data_name,
    "scenario": "difference-and-change",
    "type": "multivariate",
    "R2": r2_score(series_log_return, pred),
    "MAPE": mape(series_log_return, pred),
    "MAE": mae(series_log_return, pred),
    "RMSE": rmse(series_log_return, pred)
}

# Append a dict to the df_metrics
df_metrics = df_metrics._append(dict_metrics, ignore_index=True)

In [31]:
df_metrics.to_csv(f'outputs/04-02c-output-metrics.csv', index=False)